In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
Q1_path = os.path.join(path, 'Q1_data.csv')
df_q1 = pd.read_csv(Q1_path)

In [ ]:
# Task 2: Write your code here:
df_q1.head()

In [ ]:
# Task 3: Write your code here:
df_q1.info()

In [ ]:
# Task 4: Write your code here:
df_q1.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(df_q1['Delivery_Time'], bins=30)
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Count')
plt.title('Distribution of Delivery Time')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_q1.drop(columns=['Order_ID'], inplace=True)

In [ ]:
# Task 2: Write your code here:

missing_percentage = (df_q1.isnull().sum() / len(df_q1)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:

# lets fix them
#since the percentage is small in each of them we will use mean for nummerical
# mode for categorical since our data is small

df_q1.isnull().sum()

# Numerical columns
num_cols = df_q1.select_dtypes(include=['int64', 'float64']).columns

# Categorical columns
cat_cols = df_q1.select_dtypes(include=['object']).columns

# Fill missing numerical values
for col in num_cols:
    df_q1[col].fillna(df_q1[col].median(), inplace=True)

# Fill missing categorical values
for col in cat_cols:
    df_q1[col].fillna(df_q1[col].mode()[0], inplace=True)


In [ ]:
# Task 3: Write your code here:
#check the duplicates
df_q1.duplicated().sum()
print(df_q1.duplicated().sum())
df_q1.drop_duplicates(inplace=True)

In [ ]:
# Task 4: Write your code here:
df_q1 = pd.get_dummies(df_q1, columns=cat_cols, drop_first=True)

In [ ]:
# Task 5: Write your code here:
# (actually we have to split before scaling but since it is requred here i will solve it)
from sklearn.preprocessing import StandardScaler

X = df_q1.drop(columns=['Delivery_Time'])
y = df_q1['Delivery_Time']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df_q1, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df_q1.drop(columns=['Delivery_Time'])
y = df_q1['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)

print(f"Average MAE across folds: {np.mean(mae_scores):.2f} minutes")


In [ ]:
# Task 1: Write your code here:
from sklearn.ensemble import RandomForestRegressor

final_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y)


In [ ]:
# Task 2: Write your code here:
import matplotlib.pyplot as plt
import pandas as pd

feature_importance = pd.Series(
    final_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feature_importance.plot(kind='bar')
plt.xlabel('Features')
plt.ylabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()


In [ ]:
!pip install catboost


In [ ]:
# Task Bonus: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # --- Random Forest ---
    rf_model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)

    # --- CatBoost ---
    cb_model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        random_seed=42,
        verbose=0
    )
    cb_model.fit(X_train, y_train)

    # --- Average predictions ---
    y_pred_rf = rf_model.predict(X_val)
    y_pred_cb = cb_model.predict(X_val)

    y_pred_ensemble = (y_pred_rf + y_pred_cb) / 2

    # --- Compute MAE ---
    mae = mean_absolute_error(y_val, y_pred_ensemble)
    mae_scores.append(mae)


print(f"Average MAE across folds (Ensemble): {np.mean(mae_scores):.2f} minutes")

